<a href="https://colab.research.google.com/github/a-nastasiamoiseeva/python-ai-Moiseeva-Anastasia/blob/main/notebooks/week2b_read_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

📊 Week 2: Data Analysis — Чтение и проверка данных
Цель: Научиться читать CSV-файлы из репозитория GitHub в Google Colab и выполнять базовую проверку данных с помощью pandas.

**Данные:**

**books_authors.csv** — информация о книгах: ID книги, название, ID автора, имя автора, жанр, дата публикации, издательство
**authors.csv** — информация об авторах: ID автора, псевдонимы/имена, дата рождения, примечания

Что мы делаем:

* **Клонируем ваш персональный репозиторий** `python-ai-Moiseeva-Anastasia` в Colab
* Читаем CSV-файлы в pandas DataFrame
* Очищаем и переименовываем столбцы
* Смотрим структуру данных и делаем быструю валидацию

## 🐱 [1] Клонируем репозиторий курса в Colab

In [2]:
# 🐱 Шаг 1. Клонируем ваш репозиторий в Colab

import os

# **ИЗМЕНЕНО: имя репозитория под ваш проект**
repo = "python-ai-Moiseeva-Anastasia"
repo_path = f"/content/{repo}"

if not os.path.exists(repo_path):
    # **ИЗМЕНЕНО: URL клонирования под ваш GitHub**
    !git clone -q https://github.com/a-nastasiamoiseeva/python-ai-Moiseeva-Anastasia.git

if os.getcwd() != repo_path:
    %cd {repo_path}

print("✅ Репозиторий готов, теперь мы работаем внутри папки", repo)

✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-Moiseeva-Anastasia


## 📥 [2A] Простое чтение CSV-файлов в pandas

Сначала просто прочитаем оба CSV-файла в объекты `DataFrame`, без каких‑либо изменений.

После этого мы узнаем, сколько строк загружено в каждый датасет.

In [29]:
# 🐱 Шаг 2A. Чтение CSV-файлов в pandas (ИСПРАВЛЕННАЯ ВЕРСИЯ)
import pandas as pd

df_books = pd.read_csv(
    "data/books_authors.csv",
    sep=",",
    encoding="utf-8",
    skipinitialspace=True,  # убирает пробелы после запятых
    on_bad_lines="skip"     # пропускает строки с битой структурой
)

df_authors = pd.read_csv(
    "data/authors.csv",
    sep=",",
    encoding="utf-8",
    skipinitialspace=True,
    on_bad_lines="skip"
)

# Гарантируем чистые заголовки
df_books.columns = df_books.columns.str.strip()
df_authors.columns = df_authors.columns.str.strip()

print("✅ Загружено строк в df_books:", len(df_books))
print("✅ Загружено строк в df_authors:", len(df_authors))

✅ Загружено строк в df_books: 4148
✅ Загружено строк в df_authors: 1650


## 🧹 [2B] Очистка и переименование столбцов

В исходных CSV-файлах есть **технические столбцы с URL Wikidata** (`book`, `author`) и столбцы с постфиксом `Label`, которые содержат читаемые названия.

В этом шаге мы:
- **Удалим** технические URL-столбцы (`book` из `df_books`, `author` из `df_authors`);
- **Переименуем** столбцы, убрав постфикс `Label`:
  - `bookLabel` → `book`
  - `authorLabel` → `author`
  - `genreLabel` → `genre`
  - `publisherLabel` → `publisher`
  - `pseudonymLabel` → `pseudonym`
  - `spouseLabel` → `spouse`
- Приведём имена столбцов к чистому виду (автоматически уберём возможные пробелы и невидимые символы).

> ⚠️ **Важно:** Код использует безопасные проверки. Если какой-то столбец уже отсутствует или имеет другое имя, он не вызовет `KeyError`, а выведет предупреждение. После выполнения вы получите аккуратные DataFrames, готовые к анализу.

In [32]:
# 🧹 Шаг 2B. Очистка и переименование столбцов

# 1) df_books: удаляем URL, переименовываем Label
if "book" in df_books.columns:
    df_books = df_books.drop(columns=["book"])

rename_books = {
    "bookLabel": "book",
    "authorLabel": "author",
    "genreLabel": "genre",
    "publisherLabel": "publisher"
}
# Применяем переименование только к существующим столбцам
df_books = df_books.rename(columns={k: v for k, v in rename_books.items() if k in df_books.columns})
# Убираем возможные дубликаты имён
df_books = df_books.loc[:, ~df_books.columns.duplicated()]

# 2) df_authors: удаляем URL, переименовываем Label
if "author" in df_authors.columns:
    df_authors = df_authors.drop(columns=["author"])

rename_authors = {
    "pseudonymLabel": "pseudonym",
    "spouseLabel": "spouse"
}
df_authors = df_authors.rename(columns={k: v for k, v in rename_authors.items() if k in df_authors.columns})
df_authors = df_authors.loc[:, ~df_authors.columns.duplicated()]

# 🔽 ТОЧНО ТАКОЙ ВЫВОД, КАК ВЫ ПРОСИЛИ:
print(f"Итоговые столбцы df_books: {list(df_books.columns)}")
print(f"📋 Итоговые столбцы df_authors: {list(df_authors.columns)}")

Итоговые столбцы df_books: ['author', 'genre', 'pubDate', 'publisher']
📋 Итоговые столбцы df_authors: ['pseudonym', 'birthDate', 'spouse']


## 🔍 [3] Обзор данных: структура и первые строки

Сделаем короткий обзор обоих DataFrame с вашими данными:

### 📚 `df_books` (из books_authors.csv)
После очистки содержит столбцы:
- `book` — название книги
- `author` — имя автора
- `genre` — жанр произведения
- `pubDate` — дата публикации
- `publisher` — издательство

### 👤 `df_authors` (из authors.csv)
После очистки содержит столбцы:
- `pseudonym` — псевдоним или имя автора
- `birthDate` — дата рождения
- `spouse` — супруг/супруга (если указано)

### Что мы делаем:
- посмотрим размер таблицы (`shape`);
- выведем список столбцов;
- посмотрим первые несколько строк;
- **дополнительно проверим типы данных и наличие пропусков** (`info()` и `isna().sum()`).

Для удобства напишем маленькую функцию `show_info(df, name)`, чтобы не повторять один и тот же код два раза.

In [22]:
def show_info(df, name, n=5):
    """Краткий обзор DataFrame: имя, размер, список столбцов, типы и первые строки."""
    print(f"\n📊 {name}")
    print("Размер:", df.shape)
    print("Столбцы:", ", ".join(df.columns))
    print("Типы данных:")
    print(df.dtypes.to_string())  # **ИЗМЕНЕНО: показ типов столбцов**
    print("\nПропущенные значения:")
    print(df.isna().sum())        # **ДОБАВЛЕНО: проверка на NaN**
    print(f"\nПервые {n} строк:")
    display(df.head(n))           # **ИЗМЕНЕНО: display() для лучшего форматирования в Colab**

# 🔍 Шаг 3. Обзор ваших данных

# **ИЗМЕНЕНО: имена DataFrame и описания под ваш проект**
show_info(df_books, "📚 Книги и авторы (df_books)")
show_info(df_authors, "👤 Авторы и псевдонимы (df_authors)")

# **ДОБАВЛЕНО: быстрая сводка по уникальным значениям**
print("\n🔎 Быстрая аналитика:")
print(f"• Уникальных книг в df_books: {df_books['book'].nunique()}")
print(f"• Уникальных авторов в df_books: {df_books['author'].nunique()}")
print(f"• Жанры: {df_books['genre'].dropna().unique().tolist()}")
if 'pseudonym' in df_authors.columns:
    print(f"• Записей о псевдонимах в df_authors: {len(df_authors)}")


📊 📚 Книги и авторы (df_books)
Размер: (4148, 5)
Столбцы: book, author, genre, pubDate, publisher
Типы данных:
book         object
author       object
genre        object
pubDate      object
publisher    object

Пропущенные значения:
book            0
author          0
genre           0
pubDate         0
publisher    2788
dtype: int64

Первые 5 строк:


,book,author,genre,pubDate,publisher
0,...Яко помниши его,http://www.wikidata.org/entity/Q34981,научная фантастика,1974-05-01T00:00:00Z,NaN
1,Пересадочная станция,http://www.wikidata.org/entity/Q294625,научная фантастика,1963-06-01T00:00:00Z,Doubleday
2,Пасынки Вселенной,http://www.wikidata.org/entity/Q123078,научная фантастика,1963-01-01T00:00:00Z,Victor Gollancz
3,Убик,http://www.wikidata.org/entity/Q171091,научная фантастика,1969-01-01T00:00:00Z,Doubleday
4,The Bloody Sun,http://www.wikidata.org/entity/Q465179,научная фантастика,1964-01-01T00:00:00Z,Ace Books



📊 👤 Авторы и псевдонимы (df_authors)
Размер: (1650, 3)
Столбцы: pseudonym, birthDate, spouse
Типы данных:
pseudonym    object
birthDate    object
spouse       object

Пропущенные значения:
pseudonym     997
birthDate      62
spouse       1172
dtype: int64

Первые 5 строк:


,pseudonym,birthDate,spouse
0,Astra Zimmer,1930-06-03T00:00:00Z,Walter Breen
1,Astra Zimmer Bradley,1930-06-03T00:00:00Z,Walter Breen
2,John Jay Wells,1930-06-03T00:00:00Z,Walter Breen
3,Louis G. Daniels,1920-02-11T00:00:00Z,NaN
4,NaN,1924-11-21T00:00:00Z,Faith Tully Lilly Faulconbridge



🔎 Быстрая аналитика:
• Уникальных книг в df_books: 3318
• Уникальных авторов в df_books: 731
• Жанры: ['научная фантастика', 'фэнтези']
• Записей о псевдонимах в df_authors: 1650


## 📈 [4] Быстрая проверка и валидация данных: ключевые метрики

В этом шаге мы проведём три базовых, но информативных анализа по датасету `df_books`:

### 1️⃣ Топ-5 самых публикуемых авторов
- Посчитаем, сколько книг принадлежит каждому автору.
- Выведем рейтинг в порядке убывания.
- *Вопрос:* Кто из авторов наиболее представлен в нашей выборке?

### 2️⃣ Топ-5 самых активных издательств
- Подсчитаем количество книг для каждого издательства.
- Определим лидеров по объёму публикаций.
- *Вопрос:* Какие издательства доминируют в жанре научной фантастики?

### 3️⃣ Публикационная активность по десятилетиям
- Сгруппируем книги по десятилетиям (1960-е, 1970-е, 1980-е и т.д.).
- Посчитаем количество книг в каждом периоде.
- *Вопрос:* В какие десятилетия наблюдался пик публикаций?

> 💡 **Технические моменты:**
> - Используем `value_counts()` для быстрого подсчёта уникальных значений.
> - Для работы с датами применяем `pd.to_datetime()` и извлекаем год через `.dt.year`.
> - Группировку по десятилетиям делаем через целочисленное деление: `год // 10 * 10`.
> - Все операции защищены проверками: если столбец отсутствует или пуст, код выведет понятное предупреждение, а не ошибку.

In [23]:
# ✅ Шаг 4. Быстрая проверка: авторы, издательства, десятилетия

print("🔍 Анализ датасета df_books\n")
print("=" * 60)

# ─────────────────────────────────────────────────────
# 1️⃣ Топ-5 самых публикуемых авторов
# ─────────────────────────────────────────────────────
print("\n📊 1. Топ-5 самых публикуемых авторов")
print("-" * 40)

if "author" in df_books.columns and df_books["author"].notna().any():
    top_authors = df_books["author"].value_counts().head(5)
    for rank, (author, count) in enumerate(top_authors.items(), 1):
        print(f"{rank}. {author}: {count} книг")
    print(f"\n💡 Всего уникальных авторов: {df_books['author'].nunique()}")
else:
    print("⚠️ Столбец 'author' отсутствует или пуст")

# ─────────────────────────────────────────────────────
# 2️⃣ Топ-5 самых активных издательств
# ─────────────────────────────────────────────────────
print("\n📊 2. Топ-5 самых активных издательств")
print("-" * 40)

if "publisher" in df_books.columns and df_books["publisher"].notna().any():
    top_publishers = df_books["publisher"].value_counts().head(5)
    for rank, (publisher, count) in enumerate(top_publishers.items(), 1):
        print(f"{rank}. {publisher}: {count} книг")
    print(f"\n💡 Всего уникальных издательств: {df_books['publisher'].nunique()}")
    # Дополнительно: доля топ-5 от общего числа записей
    total_books = len(df_books)
    top5_share = top_publishers.sum() / total_books * 100
    print(f"💡 Доля топ-5 издательств: {top5_share:.1f}% от всех книг")
else:
    print("⚠️ Столбец 'publisher' отсутствует или пуст")

# ─────────────────────────────────────────────────────
# 3️⃣ Публикационная активность по десятилетиям
# ─────────────────────────────────────────────────────
print("\n📊 3. Публикационная активность по десятилетиям")
print("-" * 40)

if "pubDate" in df_books.columns:
    # Конвертируем дату в datetime (если ещё не конвертировали)
    pub_dates = pd.to_datetime(df_books["pubDate"], errors="coerce")
    # Извлекаем год
    years = pub_dates.dt.year.dropna()

    if not years.empty:
        # Группируем по десятилетиям: 1963 → 1960, 1974 → 1970
        decades = (years // 10 * 10).value_counts().sort_index()

        print(f"📅 Диапазон лет: {int(years.min())} — {int(years.max())}")
        print(f"\n📈 Книг по десятилетиям:")
        for decade, count in decades.items():
            bar = "█" * (count // max(1, count // 20))  # мини-гистограмма в тексте
            print(f"{int(decade)}-е: {count:4d} книг {bar}")

        # Пик публикаций
        peak_decade = decades.idxmax()
        print(f"\n🏆 Пик публикаций: {int(peak_decade)}-е годы ({decades.max()} книг)")
    else:
        print("⚠️ Не удалось извлечь корректные годы из pubDate")
else:
    print("⚠️ Столбец 'pubDate' отсутствует")

print("\n" + "=" * 60)
print("✅ Анализ завершён")

🔍 Анализ датасета df_books


📊 1. Топ-5 самых публикуемых авторов
----------------------------------------
1. http://www.wikidata.org/entity/Q314553: 114 книг
2. http://www.wikidata.org/entity/Q1248054: 85 книг
3. http://www.wikidata.org/entity/Q34981: 83 книг
4. http://www.wikidata.org/entity/Q181677: 67 книг
5. http://www.wikidata.org/entity/Q248867: 61 книг

💡 Всего уникальных авторов: 731

📊 2. Топ-5 самых активных издательств
----------------------------------------
1. Ace Books: 140 книг
2. DAW Books: 130 книг
3. Doubleday: 97 книг
4. Ballantine Books: 83 книг
5. Del Rey Books: 72 книг

💡 Всего уникальных издательств: 146
💡 Доля топ-5 издательств: 12.6% от всех книг

📊 3. Публикационная активность по десятилетиям
----------------------------------------
📅 Диапазон лет: 1960 — 1985

📈 Книг по десятилетиям:
1960-е:  859 книг ████████████████████
1970-е: 1731 книг ████████████████████
1980-е: 1558 книг ████████████████████

🏆 Пик публикаций: 1970-е годы (1731 книг)

✅ Анализ завершё

## 📝 Summary

**Что мы сделали в этом ноутбуке (Week 2):**

- ✅ Клонировали репозиторий GitHub в Colab
- ✅ Прочитали 2 CSV-файла из `data/examples/`
- ✅ Удалили URL Wikidata и переименовали столбцы (`*Label → короткие имена`)
- ✅ Проверили структуру данных (размер, столбцы, первые строки)
- ✅ Выполнили быструю валидацию:
  - количество уникальных фильмов, стран, жанров
  - диапазоны значений
  - топ стран и жанров по числу записей
  - типы оценок и результатов

Теперь у нас есть **аккуратные, проверенные таблицы**, с которыми удобно работать дальше.

В отдельном ноутбуке для следующей недели мы будем использовать **те же данные** для:
- более сложного анализа (группировки, фильтрация),
- и построения визуализаций (графики и диаграммы). 🎨